# YOLO11s Left/Right Hand Video Demonstration

This notebook overlays YOLO11s left/right hand detections on iAssistADL world camera videos and exports a demonstration MP4.

Classes:

- `0 = left_hand`
- `1 = right_hand`

It expects detection CSVs generated by:

```bash
cd /home/julius/projects/Identification-of-daily-activities-feature-extraction/train_YOLO_hands
python scripts/detect_world_camera_left_right_hands.py
```

In [ ]:
from pathlib import Path

import cv2
import pandas as pd
from IPython.display import Video, display

DATASET_ROOT = Path('/home/julius/datasets/seminar_object_detection')
EXPORT_ROOT = DATASET_ROOT / 'iAssistADL_dataset_export_2026-05-18'
VIDEO_ROOT = EXPORT_ROOT / 'world_camera_videos'
DETECTION_ROOT = DATASET_ROOT / 'lower_feature_outcome' / 'yolo11s_left_right_hand_outcome_csv'
DEMO_ROOT = DATASET_ROOT / 'video_demonstration' / 'yolo11s_left_right_hand'
DEMO_ROOT.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = {0: 'left_hand', 1: 'right_hand'}
COLORS = {
    0: (255, 160, 40),  # BGR, left hand
    1: (80, 220, 80),   # BGR, right hand
}

print('video root:', VIDEO_ROOT)
print('detection root:', DETECTION_ROOT)
print('demo output root:', DEMO_ROOT)

## Choose Five Recordings

Set `FULL_IDS` to up to five recording ids without `_world_camera`, for example:

```text
IA_4_H_02_Chaos_several_Tasks_combine1
```

If `FULL_IDS = None`, the notebook picks the first five available detection CSVs.

In [ ]:
FULL_IDS = None
# FULL_IDS = [
#     'IA_4_H_02_Chaos_several_Tasks_combine1',
#     'IA_4_H_02_Tasche_packen2',
#     'IA_4_H_03_Brief_schreiben2',
#     'IA_4_H_04_Chaos_several_Tasks_combine4',
#     'IA_4_H_05_Chaos_several_Task_combine4',
# ]

NUM_VIDEOS = 5

START_FRAME = 0
MAX_FRAMES = 1800       # set None to render each whole video
RENDER_STRIDE = 1       # write every Nth source frame
MIN_CONF = 0.25
LINE_THICKNESS = 2

In [ ]:
def csv_to_full_id(csv_path: Path) -> str:
    stem = csv_path.stem
    suffix = '_world_camera'
    return stem[:-len(suffix)] if stem.endswith(suffix) else stem


def infer_participant(full_id: str) -> str:
    parts = full_id.split('_')
    if len(parts) < 4:
        raise ValueError(f'Cannot infer participant from full_id: {full_id}')
    return '_'.join(parts[:4])


def find_detection_csvs(full_ids=None, num_videos=5) -> list[Path]:
    if full_ids:
        csv_paths = []
        for full_id in full_ids[:num_videos]:
            csv_path = DETECTION_ROOT / f'{full_id}_world_camera.csv'
            if not csv_path.exists():
                raise FileNotFoundError(f'Detection CSV not found: {csv_path}')
            csv_paths.append(csv_path)
        return csv_paths

    candidates = sorted(
        path for path in DETECTION_ROOT.glob('*_world_camera.csv')
        if path.name != 'yolo_hand_outcome_index.csv'
    )
    if not candidates:
        raise FileNotFoundError(f'No detection CSVs found under: {DETECTION_ROOT}')
    return candidates[:num_videos]


def find_video(full_id: str) -> Path:
    participant = infer_participant(full_id)
    video_path = VIDEO_ROOT / participant / f'{full_id}_world_camera.mp4'
    if not video_path.exists():
        raise FileNotFoundError(f'Video not found: {video_path}')
    return video_path


selected_items = []
for csv_path in find_detection_csvs(FULL_IDS, NUM_VIDEOS):
    full_id = csv_to_full_id(csv_path)
    video_path = find_video(full_id)
    selected_items.append({'full_id': full_id, 'csv_path': csv_path, 'video_path': video_path})

selected_df = pd.DataFrame(
    {
        'full_id': item['full_id'],
        'video_path': str(item['video_path']),
        'csv_path': str(item['csv_path']),
    }
    for item in selected_items
)
display(selected_df)

In [ ]:
for item in selected_items:
    detections = pd.read_csv(item['csv_path'])
    detections = detections[detections['confidence'] >= MIN_CONF].copy()
    detections['frame_id'] = detections['frame_id'].astype(int)
    detections['class_id'] = detections['class_id'].astype(int)
    item['detections'] = detections

detection_summaries = []
for item in selected_items:
    detections = item['detections']
    left_count = int((detections['class_id'] == 0).sum())
    right_count = int((detections['class_id'] == 1).sum())
    detection_summaries.append(
        {
            'full_id': item['full_id'],
            'detections': len(detections),
            'left_hand': left_count,
            'right_hand': right_count,
            'frames_with_detections': detections['frame_id'].nunique(),
        }
    )

display(pd.DataFrame(detection_summaries))
display(selected_items[0]['detections'].head())

In [ ]:
def draw_detections(frame, frame_detections, thickness=2):
    for row in frame_detections.itertuples(index=False):
        class_id = int(row.class_id)
        color = COLORS.get(class_id, (255, 255, 255))
        class_name = CLASS_NAMES.get(class_id, str(class_id))
        label = f'{class_name} {float(row.confidence):.2f}'

        x1 = int(round(float(row.x1)))
        y1 = int(round(float(row.y1)))
        x2 = int(round(float(row.x2)))
        y2 = int(round(float(row.y2)))

        h, w = frame.shape[:2]
        x1 = max(0, min(w - 1, x1))
        y1 = max(0, min(h - 1, y1))
        x2 = max(0, min(w - 1, x2))
        y2 = max(0, min(h - 1, y2))

        cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
        text_size, baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
        text_w, text_h = text_size
        y_text = max(0, y1 - text_h - baseline - 4)
        cv2.rectangle(frame, (x1, y_text), (x1 + text_w + 8, y_text + text_h + baseline + 6), color, -1)
        cv2.putText(frame, label, (x1 + 4, y_text + text_h + 2), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 2)
    return frame


def render_detection_video(video_path: Path, detections: pd.DataFrame, output_path: Path, start_frame=0, max_frames=1800, render_stride=1):
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise RuntimeError(f'Could not open video: {video_path}')

    fps = capture.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)

    output_fps = max(1.0, fps / max(1, render_stride))
    writer = cv2.VideoWriter(
        str(output_path),
        cv2.VideoWriter_fourcc(*'mp4v'),
        output_fps,
        (width, height),
    )
    if not writer.isOpened():
        capture.release()
        raise RuntimeError(f'Could not create output video: {output_path}')

    detections_by_frame = {frame_id: frame_df for frame_id, frame_df in detections.groupby('frame_id')}

    capture.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    source_frame_id = start_frame
    written = 0
    processed = 0

    while True:
        if max_frames is not None and processed >= max_frames:
            break

        ok, frame = capture.read()
        if not ok or frame is None:
            break

        if (source_frame_id - start_frame) % render_stride == 0:
            frame_detections = detections_by_frame.get(source_frame_id)
            if frame_detections is not None:
                frame = draw_detections(frame, frame_detections, thickness=LINE_THICKNESS)
            cv2.putText(frame, f'frame {source_frame_id}', (20, 36), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
            writer.write(frame)
            written += 1

        source_frame_id += 1
        processed += 1

    capture.release()
    writer.release()

    return {
        'output_path': str(output_path),
        'source_fps': fps,
        'output_fps': output_fps,
        'source_size': (width, height),
        'total_source_frames': total_frames,
        'processed_frames': processed,
        'written_frames': written,
    }

In [ ]:
render_summaries = []
for item in selected_items:
    output_video = DEMO_ROOT / f"{item['full_id']}_yolo11s_left_right_hands.mp4"
    summary = render_detection_video(
        video_path=item['video_path'],
        detections=item['detections'],
        output_path=output_video,
        start_frame=START_FRAME,
        max_frames=MAX_FRAMES,
        render_stride=RENDER_STRIDE,
    )
    item['output_video'] = output_video
    render_summaries.append({'full_id': item['full_id'], **summary})

display(pd.DataFrame(render_summaries))

In [ ]:
for item in selected_items:
    print(item['full_id'])
    print(item['output_video'])
    display(Video(str(item['output_video']), embed=True, html_attributes='controls muted'))